# Avaliação do modelo multidimensional

O star schema da **Fictoria Casa & Interiores** (fictícia, dados sintéticos) carregado no warehouse SQL Server: inventário, cardinalidades, membros especiais, grão das fatos, integridade referencial, posição do funil e a régua de SLA aplicada por intervalo. Cada seção termina numa Nota Técnica escrita a partir do observado. Contrato: [matriz de barramento](../docs/10_matriz_barramento.md); construção: [gold](../docs/11_camada_gold.md); carga: [warehouse](../docs/12_warehouse_multidimensional.md).

> Reprodutível: `uv run notebooks-modelo`.

In [1]:
import pandas as pd, pyodbc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from interiores_fictoria.config import conexao_origem, conexao_warehouse

pd.set_option("display.max_columns", 40); pd.set_option("display.width", 180)
dw = pyodbc.connect(conexao_warehouse(), timeout=15)
origem = pyodbc.connect(conexao_origem(), timeout=15)

from decimal import Decimal

def q(sql, cx=dw):
    cur = cx.cursor(); cur.execute(sql)
    cols = [d[0] for d in cur.description]
    linhas = [tuple(float(v) if isinstance(v, Decimal) else v for v in r) for r in cur.fetchall()]
    return pd.DataFrame.from_records(linhas, columns=cols)

print("warehouse:", q("SELECT DB_NAME() AS banco").iloc[0, 0], "| origem:", q("SELECT DB_NAME() AS banco", origem).iloc[0, 0])


warehouse: dw_fictoria | origem: db_fictoria


## 1. Inventário do schema `dw`: tabelas, linhas e espaço

In [2]:
display(q("""
        SELECT t.name AS tabela, SUM(p.rows) AS linhas,
               CAST(SUM(a.total_pages) * 8.0 / 1024 AS decimal(10,2)) AS mb,
               MAX(CASE WHEN i.type = 5 THEN 'columnstore' WHEN i.type = 1 THEN 'clustered (PK)' ELSE 'heap' END) AS armazenamento
        FROM sys.tables t
        JOIN sys.indexes i ON i.object_id = t.object_id AND i.index_id IN (0, 1)
        JOIN sys.partitions p ON p.object_id = t.object_id AND p.index_id = i.index_id
        JOIN sys.allocation_units a ON a.container_id = p.hobt_id OR a.container_id = p.partition_id
        WHERE t.schema_id = SCHEMA_ID('dw') GROUP BY t.name ORDER BY linhas DESC"""))

,tabela,linhas,mb,armazenamento
0,ft_orcamento_item,479670,0.07,columnstore
1,ft_funil_posicao,171060,0.07,columnstore
2,ft_follow_up,93816,0.07,columnstore
3,ft_parcela,40814,0.07,columnstore
4,ft_orcamento,28360,0.07,columnstore
5,ft_comissao,15190,0.07,columnstore
6,dim_cliente,10772,2.57,clustered (PK)
7,ft_venda,6808,0.07,columnstore
8,dim_calendario,2557,0.45,clustered (PK)
9,dim_parceiro,622,0.20,clustered (PK)


## 2. Dimensões: cardinalidade e membros especiais

In [3]:
display(q("""
        SELECT 'dim_cliente' AS dimensao, COUNT(*) AS membros, SUM(CASE WHEN sk_cliente < 0 THEN 1 ELSE 0 END) AS especiais,
               SUM(CASE WHEN fl_recorrente = 1 THEN 1 ELSE 0 END) AS recorrentes FROM dw.dim_cliente
        UNION ALL SELECT 'dim_parceiro', COUNT(*), SUM(CASE WHEN sk_parceiro < 0 THEN 1 ELSE 0 END), NULL FROM dw.dim_parceiro
        UNION ALL SELECT 'dim_vendedor', COUNT(*), SUM(CASE WHEN sk_vendedor < 0 THEN 1 ELSE 0 END), SUM(CASE WHEN fl_ativo = 1 THEN 1 ELSE 0 END) FROM dw.dim_vendedor
        UNION ALL SELECT 'dim_canal', COUNT(*), SUM(CASE WHEN sk_canal < 0 THEN 1 ELSE 0 END), NULL FROM dw.dim_canal
        UNION ALL SELECT 'dim_calendario', COUNT(*), SUM(CASE WHEN sk_data < 0 THEN 1 ELSE 0 END), SUM(CASE WHEN fl_dia_util = 1 THEN 1 ELSE 0 END) FROM dw.dim_calendario"""))

,dimensao,membros,especiais,recorrentes
0,dim_cliente,10772,1,2562.0
1,dim_parceiro,622,2,NaN
2,dim_vendedor,34,1,29.0
3,dim_canal,10,1,NaN
4,dim_calendario,2557,1,1826.0


## 3. Canal: a visão consolidada (outras origens) e o detalhe por parceiro batem por construção

In [4]:
display(q("""
        SELECT c.grupo_nome, COUNT(*) AS orcamentos, SUM(o.fl_com_parceiro) AS com_parceiro_cadastrado,
               SUM(CASE WHEN o.sk_parceiro = -2 THEN 1 ELSE 0 END) AS parceiro_texto_livre
        FROM dw.ft_orcamento o JOIN dw.dim_canal c ON c.sk_canal = o.sk_canal
        GROUP BY c.grupo_nome, c.ordem_grupo ORDER BY c.ordem_grupo"""))

,grupo_nome,orcamentos,com_parceiro_cadastrado,parceiro_texto_livre
0,Arquitetos,3092,3092,0
1,Construtoras,442,442,0
2,Indicação de clientes,3290,0,0
3,Canais próprios,6882,0,0
4,Outros,474,0,213


## 4. Grão das fatos: uma linha por chave de negócio e partições por ano

In [5]:
display(q("""
        SELECT 'ft_orcamento' AS fato, ano, COUNT(*) AS linhas, COUNT(DISTINCT sk_orcamento) AS chaves FROM dw.ft_orcamento GROUP BY ano
        UNION ALL SELECT 'ft_venda', ano, COUNT(*), COUNT(DISTINCT sk_orcamento) FROM dw.ft_venda GROUP BY ano
        ORDER BY fato, ano"""))

,fato,ano,linhas,chaves
0,ft_orcamento,2021,1486,1486
1,ft_orcamento,2022,1719,1719
2,ft_orcamento,2023,2333,2333
3,ft_orcamento,2024,2571,2571
4,ft_orcamento,2025,3517,3517
5,ft_orcamento,2026,2554,2554
6,ft_venda,2021,123,123
7,ft_venda,2022,241,241
8,ft_venda,2023,419,419
9,ft_venda,2024,629,629


## 5. Integridade referencial no warehouse: chaves órfãs (esperado: zero em todas)

In [6]:
display(q("""
        SELECT 'ft_orcamento → dim_vendedor' AS relacao, COUNT(*) AS orfas FROM dw.ft_orcamento f LEFT JOIN dw.dim_vendedor d ON d.sk_vendedor = f.sk_vendedor WHERE d.sk_vendedor IS NULL
        UNION ALL SELECT 'ft_orcamento → dim_cliente', COUNT(*) FROM dw.ft_orcamento f LEFT JOIN dw.dim_cliente d ON d.sk_cliente = f.sk_cliente WHERE d.sk_cliente IS NULL
        UNION ALL SELECT 'ft_orcamento → dim_parceiro', COUNT(*) FROM dw.ft_orcamento f LEFT JOIN dw.dim_parceiro d ON d.sk_parceiro = f.sk_parceiro WHERE d.sk_parceiro IS NULL
        UNION ALL SELECT 'ft_orcamento → dim_calendario (cadastro)', COUNT(*) FROM dw.ft_orcamento f LEFT JOIN dw.dim_calendario d ON d.sk_data = f.sk_data_cadastro WHERE d.sk_data IS NULL
        UNION ALL SELECT 'ft_venda → dim_calendario (ganho)', COUNT(*) FROM dw.ft_venda f LEFT JOIN dw.dim_calendario d ON d.sk_data = f.sk_data_ganho WHERE d.sk_data IS NULL
        UNION ALL SELECT 'ft_comissao → dim_parceiro', COUNT(*) FROM dw.ft_comissao f LEFT JOIN dw.dim_parceiro d ON d.sk_parceiro = f.sk_parceiro WHERE d.sk_parceiro IS NULL
        UNION ALL SELECT 'ft_orcamento_item → dim_item', COUNT(*) FROM dw.ft_orcamento_item f LEFT JOIN dw.dim_item d ON d.sk_item = f.sk_item WHERE d.sk_item IS NULL
        UNION ALL SELECT 'ft_funil_posicao → dim_fase', COUNT(*) FROM dw.ft_funil_posicao f LEFT JOIN dw.dim_fase d ON d.sk_fase = f.sk_fase WHERE d.sk_fase IS NULL"""))

,relacao,orfas
0,ft_orcamento → dim_vendedor,0
1,ft_orcamento → dim_cliente,0
2,ft_orcamento → dim_parceiro,0
3,ft_orcamento → dim_calendario (cadastro),0
4,ft_venda → dim_calendario (ganho),0
5,ft_comissao → dim_parceiro,0
6,ft_orcamento_item → dim_item,0
7,ft_funil_posicao → dim_fase,0


## 6. Posição do funil no último fim de mês × situação atual

In [7]:
display(q("""
        WITH atual AS (
          SELECT 'ABERTO' AS grupo_fase, SUM(fl_aberto) AS n FROM dw.ft_orcamento
          UNION ALL SELECT 'GANHO', SUM(fl_ganho) FROM dw.ft_orcamento
          UNION ALL SELECT 'PERDIDO', SUM(fl_perdido) FROM dw.ft_orcamento),
        pos AS (
          SELECT grupo_fase, SUM(qtd_orcamentos) AS n FROM dw.ft_funil_posicao
          WHERE sk_data_posicao = (SELECT MAX(sk_data_posicao) FROM dw.ft_funil_posicao) GROUP BY grupo_fase)
        SELECT a.grupo_fase, p.n AS posicao_ultimo_mes, a.n AS situacao_atual, p.n - a.n AS diferenca
        FROM atual a JOIN pos p ON p.grupo_fase = a.grupo_fase ORDER BY a.grupo_fase"""))

,grupo_fase,posicao_ultimo_mes,situacao_atual,diferenca
0,ABERTO,893,893,0
1,GANHO,3404,3404,0
2,PERDIDO,9883,9883,0


## 7. Calendário: as chaves de comparação (mesmo dia do ano anterior) resolvem para dias reais

In [8]:
display(q("""
        SELECT COUNT(*) AS dias, SUM(CASE WHEN a.sk_data IS NULL THEN 1 ELSE 0 END) AS sem_ano_anterior,
               SUM(CASE WHEN b.sk_data IS NULL THEN 1 ELSE 0 END) AS sem_dois_anos_antes
        FROM dw.dim_calendario c
        LEFT JOIN dw.dim_calendario a ON a.sk_data = c.sk_data_ano_anterior
        LEFT JOIN dw.dim_calendario b ON b.sk_data = c.sk_data_dois_anos_antes
        WHERE c.ano BETWEEN 2023 AND 2027"""))

,dias,sem_ano_anterior,sem_dois_anos_antes
0,1826,0,0


## 8. Régua de SLA aplicada por intervalo: conversão sobre fechados por ano e faixa

In [9]:
display(q("""
        WITH t AS (
          SELECT c.ano, 100.0 * SUM(o.fl_ganho) / NULLIF(SUM(o.fl_fechado), 0) AS conversao,
                 100.0 * SUM(o.fl_perdido) / NULLIF(SUM(o.fl_fechado), 0) AS perda
          FROM dw.ft_orcamento o JOIN dw.dim_calendario c ON c.sk_data = o.sk_data_cadastro
          WHERE o.fl_fechado = 1 GROUP BY c.ano)
        SELECT t.ano, CAST(t.conversao AS decimal(5,1)) AS conversao_pct, sc.faixa AS faixa_conversao,
               CAST(t.perda AS decimal(5,1)) AS perda_pct, sp.faixa AS faixa_perda
        FROM t
        JOIN dw.dim_faixa_sla sc ON sc.tipo = 'CONVERSAO' AND t.conversao > sc.limite_inferior_exclusivo AND t.conversao <= sc.limite_superior_inclusivo
        JOIN dw.dim_faixa_sla sp ON sp.tipo = 'PERDA' AND t.perda > sp.limite_inferior_exclusivo AND t.perda <= sp.limite_superior_inclusivo
        ORDER BY t.ano"""))

,ano,conversao_pct,faixa_conversao,perda_pct,faixa_perda
0,2021,8.9,Crítico,91.1,Crítico
1,2022,14.8,Ruim,85.2,Crítico
2,2023,18.5,Regular,81.5,Crítico
3,2024,24.9,Regular,75.1,Crítico
4,2025,31.4,Boa,68.6,Crítico
5,2026,49.7,Excelente,50.3,Crítico


**Nota Técnica**

- **Observado:** 22 tabelas no schema `dw` (15 dimensões e 7 fatos). 417.859 linhas de fato. 0 chaves órfãs nas relações testadas; a posição do funil no último fim de mês reproduz a situação atual e a conversão por ano cai nas faixas do SLA da diretoria (crítico em 2021. excelente em 2026).
- **Por que importa:** um star schema sem chave órfã e com grão provado é o que permite ao dashboard somar sem medo; a leitura consolidada por canal e o detalhe por parceiro saem da mesma linha. então batem por construção.
- **Ação:** o warehouse está pronto para o dashboard web e para o Power BI; a mesma carga aponta para a nuvem por `.env` quando houver destino.